In [ ]:
%pip install monai gcsfs

In [ ]:
from google.colab import auth
auth.authenticate_user()
!gcloud config set project clinimcl
!nvidia-smi

import torch, monai
print(f"[env] {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'} | torch {torch.__version__} | monai {monai.__version__}")

In [ ]:
import os, io, re, time, math, random
from collections import defaultdict
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler, autocast
from torch.serialization import add_safe_globals
import gcsfs

try:
    from monai.data.meta_tensor import MetaTensor
    from monai.utils.enums import TraceKeys
    add_safe_globals([MetaTensor, TraceKeys, np.ndarray])
except Exception:
    add_safe_globals([np.ndarray])

# ── Config ──────────────────────────────────────────────────────────
device     = "cuda" if torch.cuda.is_available() else "cpu"
BATCH      = 8
EPOCHS     = 20
IMG        = 96
LR         = 3e-4
TEMP       = 0.07
WARMUP_EP  = 3
FRAC       = 0.25
LOG_EVERY  = 50
GCS_DATA   = "gs://clinimcl-data/OASIS3/preprocessed/"
GCS_CKPT   = "gs://clinimcl-data/checkpoints/"

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# ── GCS index ──────────────────────────────────────────────────────
fs = gcsfs.GCSFileSystem(token="google_default")
pt_files = [p if p.startswith("gs://") else f"gs://{p}"
            for p in fs.ls(GCS_DATA) if p.lower().endswith(".pt")]

_re = re.compile(r"(OAS3?\d+)\D*?_d(\d+)\.pt$", re.IGNORECASE)
subjects = defaultdict(list)
for p in pt_files:
    m = _re.search(os.path.basename(p))
    if m:
        subjects[m.group(1)].append((int(m.group(2)), p))
for v in subjects.values():
    v.sort()
pairs = {s: tp for s, tp in subjects.items() if len(tp) >= 2}
pair_ids = list(pairs)
print(f"[data] {len(pt_files)} files | {len(subjects)} subjects | {len(pair_ids)} with >=2 tps")

# ── Helpers ─────────────────────────────────────────────────────────
def load_vol(url):
    with fs.open(url, "rb") as f:
        vol = torch.load(io.BytesIO(f.read()), map_location="cpu", weights_only=False)
    vol = torch.as_tensor(vol, dtype=torch.float32)
    if vol.ndim == 3: vol = vol.unsqueeze(0)
    if vol.shape[0] != 1: vol = vol[:1]
    if vol.shape[1:] != (IMG, IMG, IMG):
        vol = F.interpolate(vol.unsqueeze(0), size=(IMG,IMG,IMG),
                            mode="trilinear", align_corners=False).squeeze(0)
    return vol.contiguous()

def augment(x):
    if random.random() < 0.5:
        x = torch.flip(x, dims=[random.choice([1, 2, 3])])
    if random.random() < 0.3:
        x = x * random.uniform(0.9, 1.1) + random.uniform(-0.1, 0.1)
    if random.random() < 0.3:
        x = x + torch.randn_like(x) * random.uniform(0.01, 0.05)
    return x.clamp(0.0, 1.0)

# ── Dataset ─────────────────────────────────────────────────────────
class PairDataset(Dataset):
    def __init__(self, pairs_dict, ids):
        self.p, self.ids = pairs_dict, ids
    def __len__(self):
        return len(self.ids) * 4
    def __getitem__(self, idx):
        (_, a), (_, b) = random.sample(self.p[self.ids[idx % len(self.ids)]], 2)
        return augment(load_vol(a)), augment(load_vol(b))

def epoch_loader(ep):
    rng = random.Random(ep + 12345)
    ids = rng.sample(pair_ids, max(1, int(FRAC * len(pair_ids))))
    steps = max(60, int(math.ceil(len(pt_files) / BATCH) * FRAC))
    dl = DataLoader(PairDataset(pairs, ids), batch_size=BATCH,
                    shuffle=True, num_workers=0,
                    pin_memory=(device=="cuda"), drop_last=True)
    return dl, steps

# ── Model ──────────────────────────────────────────────────────────
class Block(nn.Module):
    def __init__(self, ci, co):
        super().__init__()
        self.net = nn.Sequential(nn.Conv3d(ci, co, 3, padding=1),
                                 nn.BatchNorm3d(co), nn.ReLU(True))
    def forward(self, x): return self.net(x)

class Encoder(nn.Module):
    def __init__(self, base=32, out=256):
        super().__init__()
        ch = [1, base, base*2, base*4, base*8]
        self.blocks = nn.ModuleList(Block(ch[i], ch[i+1]) for i in range(4))
        self.head = nn.Sequential(nn.AdaptiveAvgPool3d(1), nn.Flatten(),
                                  nn.Linear(ch[-1], out))
    def forward(self, x):
        for b in self.blocks:
            x = b(F.max_pool3d(x, 2))
        return self.head(x)

class ClinImCL(nn.Module):
    def __init__(self, proj=128):
        super().__init__()
        self.enc = Encoder()
        self.proj = nn.Sequential(nn.Linear(256, 256), nn.ReLU(True),
                                  nn.Linear(256, proj))
    def forward(self, x):
        h = self.enc(x)
        return F.normalize(self.proj(h), dim=1), h

model = ClinImCL().to(device)
print(f"[model] {sum(p.numel() for p in model.parameters())/1e6:.2f}M params")

def info_nce(z1, z2):
    z1, z2 = F.normalize(z1, dim=1), F.normalize(z2, dim=1)
    logits = (z1 @ z2.t()) / TEMP
    tgt = torch.arange(z1.size(0), device=z1.device)
    return 0.5 * (F.cross_entropy(logits, tgt) + F.cross_entropy(logits.t(), tgt))

# ── Optimizer + schedule ───────────────────────────────────────────
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scaler = GradScaler("cuda", enabled=(device=="cuda"))
steps_ep = max(60, int(math.ceil(len(pt_files)/BATCH) * FRAC))
wu = WARMUP_EP * steps_ep
total = EPOCHS * steps_ep

def lr_fn(s):
    if s < wu: return s / max(1, wu)
    return 0.5 * (1 + math.cos(math.pi * (s - wu) / max(1, total - wu)))
sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_fn)

# ── Train ──────────────────────────────────────────────────────────
gs = 0
t0_all = time.time()
for ep in range(1, EPOCHS + 1):
    dl, max_steps = epoch_loader(ep)
    model.train()
    losses, t0, step = [], time.time(), 0

    for x1, x2 in dl:
        x1, x2 = x1.to(device, non_blocking=True), x2.to(device, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with autocast(device_type="cuda", enabled=(device=="cuda")):
            loss = info_nce(model(x1)[0], model(x2)[0])
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update(); sched.step()
        losses.append(loss.item()); gs += 1; step += 1
        if LOG_EVERY and gs % LOG_EVERY == 0:
            print(f"  [ep {ep:02d} step {gs:05d}] loss={np.mean(losses[-50:]):.4f} lr={sched.get_last_lr()[0]:.2e}")
        if step >= max_steps: break

    print(f"[ep {ep:02d}] {time.time()-t0:.0f}s | steps={step} | loss={np.mean(losses):.4f}")

    path = f"/content/clinimcl_ep{ep:02d}.pth"
    torch.save({"epoch": ep, "model": model.state_dict(),
                "optimizer": opt.state_dict(), "scheduler": sched.state_dict(),
                "cfg": dict(IMG=IMG, TEMP=TEMP, proj=128, base=32, FRAC=FRAC, BATCH=BATCH)}, path)
    dst = f"{GCS_CKPT}clinimcl_ep{ep:02d}_{time.strftime('%Y%m%d_%H%M%S')}.pth"
    if os.system(f"gsutil cp {path} {dst}") == 0:
        os.remove(path)
        print(f"[upload] {dst}")
    else:
        print(f"[save] {path}")

print(f"\nDone in {time.time()-t0_all:.0f}s")